# YOLOv8-OBB Training: Bottle vs Can Detection

**Goal:** Train a YOLOv8n-OBB model on the Roboflow bottle-and-can dataset using Google Colab T4 GPU.

**Dataset:** Roboflow `bouteille/bottle-and-can` v7, re-split for class balance  
All splits balanced: bottle ~45% / can ~55%

**Model:** YOLOv8n-OBB (nano, ~3M params) — oriented bounding boxes, transfer learning from COCO pretrained weights

**Training plan:** 100 epochs, imgsz=640, batch=16, AdamW optimizer, cosine LR

## 1. Environment Setup

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## 2. Download Dataset from Roboflow

In [ ]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key="l2vJEUtQX03ZwuWPpsXi")
project = rf.workspace("bouteille").project("bottle-and-can")
dataset = project.version(7).download("yolov8-obb", location="/content/raw")
print("Downloaded to:", dataset.location)

## 3. Re-split for Class Balance

Roboflow v7 valid split is 93% bottle / 7% can — this causes the model to classify
everything as bottle. We re-split all images into a balanced 80/10/10 train/val/test.

In [ ]:
import random
import shutil
from collections import defaultdict, Counter
from pathlib import Path

random.seed(42)

SRC = Path("/content/raw")
DST = Path("/content/dataset")
RAW_SPLITS = ["train", "valid", "test"]

def dominant_class(label_path):
    counts = defaultdict(int)
    for line in label_path.read_text().strip().splitlines():
        if line:
            counts[int(line.split()[0])] += 1
    return max(counts, key=counts.__getitem__) if counts else None

# Collect all images grouped by dominant class
by_class = defaultdict(list)
for split in RAW_SPLITS:
    for img in (SRC / split / "images").glob("*.jpg"):
        lbl = SRC / split / "labels" / (img.stem + ".txt")
        if lbl.exists():
            cls = dominant_class(lbl)
            if cls is not None:
                by_class[cls].append((img, lbl))

print("Raw counts:", {c: len(v) for c, v in by_class.items()})

# Balance by capping at smaller class size
n = min(len(v) for v in by_class.values())
balanced = []
for cls, pairs in by_class.items():
    random.shuffle(pairs)
    balanced.extend(pairs[:n])
random.shuffle(balanced)

total = len(balanced)
n_test  = max(50, int(total * 0.10))
n_val   = max(50, int(total * 0.10))
n_train = total - n_test - n_val

splits_out = {
    "train": balanced[:n_train],
    "valid": balanced[n_train:n_train + n_val],
    "test":  balanced[n_train + n_val:],
}

if DST.exists():
    shutil.rmtree(DST)

for split, pairs in splits_out.items():
    (DST / split / "images").mkdir(parents=True, exist_ok=True)
    (DST / split / "labels").mkdir(parents=True, exist_ok=True)
    for img, lbl in pairs:
        shutil.copy(img, DST / split / "images" / img.name)
        shutil.copy(lbl, DST / split / "labels" / lbl.name)

# Write data.yaml
(DST / "data.yaml").write_text(
    "path: /content/dataset\n"
    "train: train/images\n"
    "val: valid/images\n"
    "test: test/images\n\n"
    "names:\n"
    "  0: bottle\n"
    "  1: can\n\n"
    "nc: 2\n"
)

print(f"\nBalanced split — train:{n_train}  val:{n_val}  test:{n_test}")
for split in ["train", "valid", "test"]:
    c = Counter()
    for lbl in (DST / split / "labels").glob("*.txt"):
        for line in lbl.read_text().strip().splitlines():
            if line:
                c[int(line.split()[0])] += 1
    t = sum(c.values())
    print(f"  {split:5s}: bottle={c[0]} ({100*c[0]//max(t,1)}%)  can={c[1]} ({100*c[1]//max(t,1)}%)")

## 4. Sanity Check — Visualize Sample Annotations

In [ ]:
import cv2, glob, random
import numpy as np
import matplotlib.pyplot as plt

CLASS_NAMES = {0: "bottle", 1: "can"}
COLORS = {0: (0, 200, 0), 1: (0, 100, 255)}

def draw_obb(img, label_path):
    h, w = img.shape[:2]
    if not os.path.exists(label_path):
        return img
    for line in open(label_path):
        parts = line.strip().split()
        if len(parts) < 9:
            continue
        cls = int(parts[0])
        coords = list(map(float, parts[1:9]))
        pts = np.array([[coords[i]*w, coords[i+1]*h] for i in range(0, 8, 2)], dtype=np.int32)
        cv2.polylines(img, [pts], True, COLORS.get(cls, (200,200,200)), 3)
        cv2.putText(img, CLASS_NAMES.get(cls, str(cls)), tuple(pts[0]),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, COLORS.get(cls, (200,200,200)), 2)
    return img

random.seed(42)
all_imgs = glob.glob("/content/dataset/train/images/*.jpg")
sample_imgs = random.sample(all_imgs, min(6, len(all_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, sample_imgs):
    lbl = img_path.replace("/images/", "/labels/").replace(".jpg", ".txt")
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    img = draw_obb(img, lbl)
    ax.imshow(img)
    ax.axis("off")
plt.suptitle("Sample OBB annotations (green=bottle, orange=can)", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-obb.pt")

results = model.train(
    data="/content/dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    patience=20,
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    project="/content/runs",
    name="bottle_can_obb_v3",
    exist_ok=True,
    plots=True,
)

## 6. Evaluate on Test Set

In [ ]:
best = "/content/runs/bottle_can_obb_v3/weights/best.pt"
model = YOLO(best)
metrics = model.val(data="/content/dataset/data.yaml", split="test", plots=True)

print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
per_class = dict(zip(["bottle", "can"], metrics.box.maps))
print(f"bottle mAP@0.5: {per_class['bottle']:.4f}")
print(f"can    mAP@0.5: {per_class['can']:.4f}")

## 7. Save to Google Drive

In [ ]:
from google.colab import drive
import shutil, os

drive.mount("/content/drive")

DRIVE_PROJECT = "/content/drive/MyDrive/yolo-portfolio"
os.makedirs(f"{DRIVE_PROJECT}/models", exist_ok=True)

shutil.copy(best, f"{DRIVE_PROJECT}/models/best.pt")
print("Weights saved:", f"{DRIVE_PROJECT}/models/best.pt")

if os.path.exists(f"{DRIVE_PROJECT}/results/training"):
    shutil.rmtree(f"{DRIVE_PROJECT}/results/training")
shutil.copytree("/content/runs/bottle_can_obb_v3", f"{DRIVE_PROJECT}/results/training")
print("Training results saved to Drive.")

## 8. Save Sample Predictions

In [ ]:
os.makedirs("/content/preds", exist_ok=True)

model.predict(
    source="/content/dataset/test/images",
    save=True,
    conf=0.25,
    project="/content/preds",
    name="test_predictions",
    exist_ok=True,
)

shutil.copytree(
    "/content/preds/test_predictions",
    f"{DRIVE_PROJECT}/results/sample_predictions/test",
    dirs_exist_ok=True,
)
print("Sample predictions saved to Drive.")